# Unified Higher-Order Gaussian Action Corrections

This notebook treats aberrated lenses and thin atomic slices as the same operation: a nonlinear local action `S(x, y)` applied to a Gaussian beam. The goal is to compare candidate higher-order corrections and decide which one is usable for both smooth higher-order lenses and sharp atomic potentials.

The central candidate is sampled fitted quadratic action with residual-driven split/refit. Each Gaussian sees the best local quadratic fit to `S(x, y)` over its own support. The remaining non-quadratic residual is measured in radians, and beams are split only when that residual is too large.

The Hermite first-order residual correction is included for the smooth lens case as a best-case comparator, not as the assumed atom method.

## Setup

The default sizes below are intentionally moderate. Increase `GRID_SHAPE`, `LENS_SWEEP`, `ATOM_DZ_SWEEP`, or `SPLIT_GRID_SHAPE` after the method ranking is clear.

In [ ]:
import os
import time
from collections import defaultdict
from dataclasses import dataclass

os.environ.setdefault("JAX_ENABLE_X64", "1")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.4")

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from temgym_core.aberrations import KrivanekCoeffs
from temgym_core.components import Detector, KrivanekLens
from temgym_core.constants import energy2wavelength
from temgym_core.evaluate import evaluate_gaussians_fast, evaluate_gaussians_for
from temgym_core.gaussian import (
    FreeSpacePropagator,
    GaussianBeam,
    make_gaussian,
)
from temgym_core.gaussian_corrections import (
    apply_fitted_quadratic_action,
    apply_taylor_phase_action,
    concatenate_gaussian_beams,
    first_order_phase_residual_correction_factor,
    fit_phase_residual_polynomial_on_beam,
    sampled_quadratic_residual_on_beam,
    split_gaussian_realspace,
)
from temgym_core.potential import potential_smoothed_from_r2
from temgym_core.utils import FresnelPropagator, zero_phase

jax.config.update("jax_enable_x64", True)

GRID_SHAPE = 96
EVAL_METHOD = "scan"  # use "auto" on a working GPU/Pallas setup
RESIDUAL_TARGET_RAD = 0.25
RESIDUAL_BAND_RAD = (0.1, 0.3)
FIT_SUPPORT_RADIUS = 1.0
FIT_SAMPLES_PER_AXIS = 5
SPLIT_GRID_SHAPE = (5, 5)
SPLIT_CHILD_WIDTH_SCALE = 0.6

print("JAX backend:", jax.default_backend())

## Shared Experiment Helpers

Every method below receives the same local action function. The reference always applies the exact sampled transmission `exp(i k S)` on a grid, then propagates with Fresnel propagation. The Gaussian methods approximate that same action on each Gaussian.

In [ ]:
@dataclass(frozen=True)
class MethodResult:
    name: str
    field: jnp.ndarray
    beams: int
    time_s: float
    residual_max_rad: float
    residual_p90_rad: float


def block_until_ready(value):
    def block_leaf(x):
        return x.block_until_ready() if hasattr(x, "block_until_ready") else x

    return jax.tree_util.tree_map(block_leaf, value)


def timed_call(fn):
    t0 = time.perf_counter()
    value = fn()
    block_until_ready(value)
    return value, time.perf_counter() - t0


def beam_count(beam: GaussianBeam) -> int:
    return int(np.asarray(beam.to_vector().x).size)


def beam_iter(beam: GaussianBeam):
    vector = beam.to_vector()
    for i in range(beam_count(vector)):
        yield vector[i]


def concatenate_or_single(beams):
    if len(beams) == 1:
        return beams[0].to_vector()
    return concatenate_gaussian_beams(beams).to_vector()


def evaluate_field(beam, grid):
    if EVAL_METHOD == "loop":
        return evaluate_gaussians_for(beam.to_vector(), grid)
    return evaluate_gaussians_fast(beam.to_vector(), grid, method=EVAL_METHOD)


def aligned_field_error(test, ref):
    alpha = jnp.vdot(test, ref) / jnp.vdot(test, test)
    return float(jnp.linalg.norm(alpha * test - ref) / jnp.linalg.norm(ref))


def normalized_intensity_error(test, ref):
    test_i = jnp.abs(test) ** 2
    ref_i = jnp.abs(ref) ** 2
    test_i = test_i / jnp.sum(test_i)
    ref_i = ref_i / jnp.sum(ref_i)
    return float(jnp.linalg.norm(test_i - ref_i) / jnp.linalg.norm(ref_i))


def center_zero_phase(field):
    return zero_phase(field, field.shape[0] // 2, field.shape[1] // 2)


def residual_values_rad(beam, action_fn, *, support_radius=1.0, fit_samples_per_axis=5):
    values = []
    for ray in beam_iter(beam):
        residual = sampled_quadratic_residual_on_beam(
            action_fn,
            ray,
            scale=ray.k,
            support_radius=support_radius,
            fit_support_radius=support_radius,
            fit_samples_per_axis=fit_samples_per_axis,
        )
        values.append(float(residual.max_abs))
    return np.asarray(values, dtype=float)


def residual_summary(beam, action_fn):
    values = residual_values_rad(
        beam,
        action_fn,
        support_radius=FIT_SUPPORT_RADIUS,
        fit_samples_per_axis=FIT_SAMPLES_PER_AXIS,
    )
    return {
        "max": float(np.max(values)),
        "p90": float(np.quantile(values, 0.9)),
        "median": float(np.median(values)),
    }


def grid_reference_field(beam, grid, action_fn, propagation_distance):
    vector = beam.to_vector()
    wavelength = vector.wavelength[0]
    k = vector.k[0]
    input_field = evaluate_field(beam, grid)
    action_grid = jax.vmap(action_fn)(grid.coords).reshape(grid.shape)
    transmitted = input_field * jnp.exp(1j * k * action_grid)
    return FresnelPropagator(
        transmitted,
        L=grid.pixel_size[0] * grid.shape[0],
        wavelength=wavelength,
        z=propagation_distance,
    )


def apply_taylor_bundle(beam, action_fn):
    return concatenate_or_single([
        apply_taylor_phase_action(ray, action_fn) for ray in beam_iter(beam)
    ])


def apply_fitted_bundle(beam, action_fn):
    return concatenate_or_single([
        apply_fitted_quadratic_action(
            ray,
            action_fn,
            fit_support_radius=FIT_SUPPORT_RADIUS,
            fit_samples_per_axis=FIT_SAMPLES_PER_AXIS,
        )
        for ray in beam_iter(beam)
    ])


def apply_split_then_fitted_bundle(beam, action_fn):
    out = []
    for ray in beam_iter(beam):
        children = split_gaussian_realspace(
            ray,
            grid_shape=SPLIT_GRID_SHAPE,
            support_radius=1.1,
            child_width_scale=SPLIT_CHILD_WIDTH_SCALE,
            fit_samples_per_axis=7,
        )
        out.extend(
            apply_fitted_quadratic_action(
                child,
                action_fn,
                fit_support_radius=FIT_SUPPORT_RADIUS,
                fit_samples_per_axis=FIT_SAMPLES_PER_AXIS,
            )
            for child in beam_iter(children)
        )
    return concatenate_or_single(out)


def apply_adaptive_fitted_bundle(beam, action_fn, *, residual_target=RESIDUAL_TARGET_RAD):
    out = []
    for ray in beam_iter(beam):
        residual = sampled_quadratic_residual_on_beam(
            action_fn,
            ray,
            scale=ray.k,
            support_radius=FIT_SUPPORT_RADIUS,
            fit_samples_per_axis=FIT_SAMPLES_PER_AXIS,
        )
        if float(residual.max_abs) > residual_target:
            parents = split_gaussian_realspace(
                ray,
                grid_shape=SPLIT_GRID_SHAPE,
                support_radius=1.1,
                child_width_scale=SPLIT_CHILD_WIDTH_SCALE,
                fit_samples_per_axis=7,
            )
        else:
            parents = ray
        out.extend(
            apply_fitted_quadratic_action(
                child,
                action_fn,
                fit_support_radius=FIT_SUPPORT_RADIUS,
                fit_samples_per_axis=FIT_SAMPLES_PER_AXIS,
            )
            for child in beam_iter(parents)
        )
    return concatenate_or_single(out)


def propagated_method_field(beam, grid, action_fn, propagation_distance, apply_method):
    after_action = apply_method(beam, action_fn)
    propagated = FreeSpacePropagator()(after_action, propagation_distance)
    return evaluate_field(propagated, grid), beam_count(propagated)


def hermite_lens_field(beam, grid, action_fn, propagation_distance):
    if beam_count(beam) != 1:
        raise ValueError("The Hermite diagnostic is implemented here for one Gaussian.")
    ray = next(beam_iter(beam))
    taylor_ray = apply_taylor_phase_action(ray, action_fn)
    propagated = FreeSpacePropagator()(taylor_ray, propagation_distance)
    base_field = evaluate_field(propagated, grid)
    fit = fit_phase_residual_polynomial_on_beam(
        action_fn,
        ray,
        min_degree=3,
        max_degree=4,
        fit_support_radius=FIT_SUPPORT_RADIUS,
        fit_samples_per_axis=7,
    )
    factor = first_order_phase_residual_correction_factor(
        taylor_ray,
        grid.coords,
        propagation_distance,
        fit,
    ).reshape(grid.shape)
    return base_field * (1.0 + factor), beam_count(propagated)


def run_action_case(case_name, beam, grid, action_fn, propagation_distance, methods):
    reference, t_ref = timed_call(
        lambda: grid_reference_field(beam, grid, action_fn, propagation_distance)
    )
    residual = residual_summary(beam, action_fn)
    records = []
    fields = {"reference": reference}
    for method_name, method_kind in methods:
        if method_kind == "taylor":
            fn = lambda: propagated_method_field(
                beam, grid, action_fn, propagation_distance, apply_taylor_bundle
            )
        elif method_kind == "fitted":
            fn = lambda: propagated_method_field(
                beam, grid, action_fn, propagation_distance, apply_fitted_bundle
            )
        elif method_kind == "split_fitted":
            fn = lambda: propagated_method_field(
                beam, grid, action_fn, propagation_distance, apply_split_then_fitted_bundle
            )
        elif method_kind == "adaptive":
            fn = lambda: propagated_method_field(
                beam, grid, action_fn, propagation_distance, apply_adaptive_fitted_bundle
            )
        elif method_kind == "hermite":
            fn = lambda: hermite_lens_field(beam, grid, action_fn, propagation_distance)
        else:
            raise ValueError(f"Unknown method kind {method_kind!r}")

        (field, beams), t_method = timed_call(fn)
        fields[method_name] = field
        records.append(
            {
                "case": case_name,
                "method": method_name,
                "field_error": aligned_field_error(field, reference),
                "intensity_error": normalized_intensity_error(field, reference),
                "beams": beams,
                "time_s": t_method,
                "reference_time_s": t_ref,
                "residual_max_rad": residual["max"],
                "residual_p90_rad": residual["p90"],
                "residual_median_rad": residual["median"],
            }
        )
    return records, fields


def print_records(records, sort_keys=("case", "method")):
    for rec in sorted(records, key=lambda r: tuple(r[k] for k in sort_keys)):
        print(
            f"{rec['case']:34s} | {rec['method']:14s} | "
            f"field={rec['field_error']:.3g} | intensity={rec['intensity_error']:.3g} | "
            f"resid_max={rec['residual_max_rad']:.3g} rad | "
            f"beams={rec['beams']:4d} | time={rec['time_s']:.3f}s"
        )

## Focused One-Method Test

This is the compact test of the proposed single method: **adaptive sampled fitted quadratic action**. It is run on two deliberately hard examples, a strongly aberrated Krivanek lens and a centered Au atom thin slice. Taylor is included only as a baseline so the improvement is visible.

In [ ]:
FOCUSED_AU_LOBATO = jnp.array(
    [
        [
            1.6759346706487,
            3.00486602969729,
            0.595340013161635,
            0.0117163186623094,
            4.296782976398e-05,
        ],
        [
            5.52231093211402,
            1.38007223007196,
            0.162229237655945,
            0.00901814890416575,
            0.00037927766747767,
        ],
    ],
    dtype=jnp.float64,
)


def focused_kriv_coeffs(scale=1.0):
    return KrivanekCoeffs(
        C21=scale * 300.0,
        phi21=0.25 * jnp.pi,
        C23=scale * 120.0,
        phi23=-0.1 * jnp.pi,
        C32=scale * 150.0,
        phi32=0.4 * jnp.pi,
        C43=scale * 150.0,
        phi43=0.2 * jnp.pi,
    )


def focused_multi_atom_action(xy, z, sigma, k, atom_xyz, dz, cutoff_radius):
    def one_atom(atom):
        r2 = (
            (xy[0] - atom[0]) ** 2
            + (xy[1] - atom[1]) ** 2
            + (z - atom[2]) ** 2
        )
        return potential_smoothed_from_r2(r2, FOCUSED_AU_LOBATO, cutoff_radius)

    potential = jnp.sum(jax.vmap(one_atom)(atom_xyz))
    return -(sigma / k) * potential * dz


def focused_atom_action_for_beam(beam, atom_xyz, dz, cutoff_radius):
    vector = beam.to_vector()
    sigma = vector.sigma[0]
    k = vector.k[0]
    z = vector.z[0]
    return lambda xy: focused_multi_atom_action(
        xy, z, sigma, k, atom_xyz, dz, cutoff_radius
    )


FOCUSED_METHODS = [
    ("Taylor baseline", "taylor"),
    ("Universal adaptive fit", "adaptive"),
]

focused_records = []
focused_fields = {}

# Hard lens: one Gaussian probe through a strong higher-order Krivanek phase.
focused_lens_voltage = 300e3
focused_lens_focal = 5e-3
focused_lens_width = 2.0e-6
focused_lens_pixel = focused_lens_width / GRID_SHAPE
focused_lens_grid = Detector(
    z=focused_lens_focal,
    pixel_size=(focused_lens_pixel, focused_lens_pixel),
    shape=(GRID_SHAPE, GRID_SHAPE),
)
focused_lens_probe = make_gaussian(
    x=0.0,
    y=0.0,
    z=focused_lens_focal,
    voltage=focused_lens_voltage,
    waist_x=0.03e-6,
    waist_y=0.03e-6,
)
focused_lens = KrivanekLens(
    z=focused_lens_focal,
    focal_length=focused_lens_focal,
    coeffs=focused_kriv_coeffs(scale=1.0),
)
lens_demo_records, lens_demo_fields = run_action_case(
    "hard Krivanek lens",
    focused_lens_probe,
    focused_lens_grid,
    lambda xy: focused_lens.phase_shift(xy),
    focused_lens_focal,
    FOCUSED_METHODS,
)
for rec in lens_demo_records:
    rec["domain"] = "focused_lens"
    rec["sweep_value"] = 1.0
focused_records.extend(lens_demo_records)
focused_fields["hard Krivanek lens"] = lens_demo_fields

# Hard atom: one Gaussian centered on a sharp Au atom core.
focused_atom_extent = 3.0
focused_atom_pixel = 2 * focused_atom_extent / GRID_SHAPE
focused_atom_grid = Detector(
    z=0.0,
    pixel_size=(focused_atom_pixel, focused_atom_pixel),
    shape=(GRID_SHAPE, GRID_SHAPE),
)
focused_atom_beam = make_gaussian(
    x=0.0,
    y=0.0,
    z=0.0,
    voltage=200e3,
    waist_x=0.5,
    waist_y=0.5,
    wavelength_unit="angstrom",
)
focused_atom_xyz = jnp.array([[0.0, 0.0, 0.0]], dtype=jnp.float64)
focused_atom_dz = 0.02
focused_atom_action = focused_atom_action_for_beam(
    focused_atom_beam,
    focused_atom_xyz,
    focused_atom_dz,
    focused_atom_pixel / 3.0,
)
atom_demo_records, atom_demo_fields = run_action_case(
    "centered Au atom",
    focused_atom_beam,
    focused_atom_grid,
    focused_atom_action,
    2.0,
    FOCUSED_METHODS,
)
for rec in atom_demo_records:
    rec["domain"] = "focused_atom"
    rec["sweep_value"] = focused_atom_dz
focused_records.extend(atom_demo_records)
focused_fields["centered Au atom"] = atom_demo_fields

print_records(focused_records)

In [ ]:
def plot_focused_method_summary(records):
    cases = list(dict.fromkeys(r["case"] for r in records))
    methods = list(dict.fromkeys(r["method"] for r in records))
    x = np.arange(len(cases))
    width = 0.34

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
    for j, method in enumerate(methods):
        rows = [next(r for r in records if r["case"] == case and r["method"] == method) for case in cases]
        offset = (j - 0.5 * (len(methods) - 1)) * width
        axes[0].bar(x + offset, [r["field_error"] for r in rows], width, label=method)
        axes[1].bar(x + offset, [r["beams"] for r in rows], width, label=method)
        axes[2].bar(x + offset, [r["time_s"] for r in rows], width, label=method)

    axes[0].set_yscale("log")
    axes[2].set_yscale("log")
    axes[0].set_ylabel("aligned field error")
    axes[1].set_ylabel("output beams")
    axes[2].set_ylabel("runtime (s)")
    for ax in axes:
        ax.set_xticks(x, cases, rotation=20, ha="right")
        ax.grid(True, axis="y", alpha=0.3)
    axes[0].legend(fontsize=8)
    fig.suptitle("One-method test: adaptive fitted action vs Taylor baseline")
    fig.tight_layout()
    return fig


def show_focused_case(case, fields, extent):
    reference = np.asarray(fields["reference"])
    taylor = np.asarray(fields["Taylor baseline"])
    universal = np.asarray(fields["Universal adaptive fit"])
    images = [
        (np.abs(reference) ** 2, "reference intensity", "inferno"),
        (np.abs(taylor) ** 2, "Taylor intensity", "inferno"),
        (np.abs(universal) ** 2, "universal intensity", "inferno"),
        (
            np.abs(taylor) ** 2 - np.abs(reference) ** 2,
            "Taylor - reference",
            "coolwarm",
        ),
        (
            np.abs(universal) ** 2 - np.abs(reference) ** 2,
            "universal - reference",
            "coolwarm",
        ),
        (
            np.angle(universal * np.conj(reference)),
            "universal relative phase",
            "twilight",
        ),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(10.5, 6.0), sharex=True, sharey=True)
    diff_limit = max(np.max(np.abs(images[3][0])), np.max(np.abs(images[4][0])))
    for ax, (data, title, cmap) in zip(axes.ravel(), images):
        kwargs = {}
        if " - reference" in title and diff_limit > 0:
            kwargs.update(vmin=-diff_limit, vmax=diff_limit)
        im = ax.imshow(data, extent=extent, origin="lower", cmap=cmap, **kwargs)
        ax.set_title(title)
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        fig.colorbar(im, ax=ax, shrink=0.72)
    fig.suptitle(case)
    fig.tight_layout()
    return fig


plot_focused_method_summary(focused_records)
show_focused_case(
    "hard Krivanek lens",
    focused_fields["hard Krivanek lens"],
    tuple(float(v) * 1e6 for v in focused_lens_grid.extent),
)
show_focused_case(
    "centered Au atom",
    focused_fields["centered Au atom"],
    tuple(float(v) for v in focused_atom_grid.extent),
)
plt.show()

## Lens Study: Krivanek Aberration Sweep

This sweep uses one focused Gaussian probe at the lens plane. The reference applies the exact aberrated phase mask on the grid and then propagates by Fresnel propagation. The Gaussian methods see only the local action function.

In [ ]:
def make_kriv_coeffs(scale):
    return KrivanekCoeffs(
        C21=scale * 300.0,
        phi21=0.25 * jnp.pi,
        C23=scale * 120.0,
        phi23=-0.1 * jnp.pi,
        C32=scale * 150.0,
        phi32=0.4 * jnp.pi,
        C43=scale * 150.0,
        phi43=0.2 * jnp.pi,
    )


lens_voltage = 300e3
lens_focal_length = 5e-3
lens_window_width = 2.0e-6
lens_pixel = lens_window_width / GRID_SHAPE
lens_grid = Detector(
    z=lens_focal_length,
    pixel_size=(lens_pixel, lens_pixel),
    shape=(GRID_SHAPE, GRID_SHAPE),
)
lens_probe = make_gaussian(
    x=0.0,
    y=0.0,
    z=lens_focal_length,
    voltage=lens_voltage,
    waist_x=0.03e-6,
    waist_y=0.03e-6,
)

LENS_SWEEP = [0.0, 0.25, 0.5, 1.0]
LENS_METHODS = [
    ("Taylor", "taylor"),
    ("Fitted quadratic", "fitted"),
    ("Split + fitted", "split_fitted"),
    ("Adaptive fitted", "adaptive"),
    ("Hermite residual", "hermite"),
]

lens_records = []
lens_fields_by_case = {}
for scale in LENS_SWEEP:
    lens = KrivanekLens(
        z=lens_focal_length,
        focal_length=lens_focal_length,
        coeffs=make_kriv_coeffs(scale),
    )
    case_name = f"lens scale={scale:g}"
    records, fields = run_action_case(
        case_name,
        lens_probe,
        lens_grid,
        lambda xy, lens=lens: lens.phase_shift(xy),
        lens_focal_length,
        LENS_METHODS,
    )
    for rec in records:
        rec["domain"] = "lens"
        rec["sweep_value"] = scale
    lens_records.extend(records)
    lens_fields_by_case[case_name] = fields

print_records(lens_records)

In [ ]:
def plot_error_sweep(records, title, xlabel):
    grouped = defaultdict(list)
    for rec in records:
        grouped[rec["method"]].append(rec)

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharex=True)
    for method, rows in grouped.items():
        rows = sorted(rows, key=lambda r: r["sweep_value"])
        x = [r["sweep_value"] for r in rows]
        axes[0].plot(x, [r["field_error"] for r in rows], marker="o", label=method)
        axes[1].plot(x, [r["intensity_error"] for r in rows], marker="o", label=method)
    for ax, ylabel in zip(axes, ["aligned field error", "normalized intensity error"]):
        ax.set_yscale("log")
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.3)
    axes[1].legend(fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()
    return fig


plot_error_sweep(lens_records, "Krivanek lens correction sweep", "aberration scale")
plt.show()

## Atom Setup

The atomic examples use a thin-slice action from a projected Lobato-style potential. Coordinates and propagation distances are in angstroms because the input beams use `wavelength_unit="angstrom"`.

In [ ]:
AU_LOBATO = jnp.array(
    [
        [
            1.6759346706487,
            3.00486602969729,
            0.595340013161635,
            0.0117163186623094,
            4.296782976398e-05,
        ],
        [
            5.52231093211402,
            1.38007223007196,
            0.162229237655945,
            0.00901814890416575,
            0.00037927766747767,
        ],
    ],
    dtype=jnp.float64,
)


def multi_atom_action(xy, z, sigma, k, atom_xyz, dz, cutoff_radius, element_params):
    def one_atom(atom):
        r2 = (
            (xy[0] - atom[0]) ** 2
            + (xy[1] - atom[1]) ** 2
            + (z - atom[2]) ** 2
        )
        return potential_smoothed_from_r2(r2, element_params, cutoff_radius)

    potential = jnp.sum(jax.vmap(one_atom)(atom_xyz))
    return -(sigma / k) * potential * dz


def atom_plane(n_side, spacing):
    axis = (jnp.arange(n_side, dtype=jnp.float64) - 0.5 * (n_side - 1)) * spacing
    xx, yy = jnp.meshgrid(axis, axis, indexing="ij")
    return jnp.stack(
        (xx.ravel(), yy.ravel(), jnp.zeros((n_side * n_side,), dtype=jnp.float64)),
        axis=-1,
    )


def make_atom_action_for_beam(beam, atom_xyz, dz, cutoff_radius, element_params=AU_LOBATO):
    vector = beam.to_vector()
    sigma = vector.sigma[0]
    k = vector.k[0]
    z = vector.z[0]
    return lambda xy: multi_atom_action(
        xy,
        z,
        sigma,
        k,
        atom_xyz,
        dz,
        cutoff_radius,
        element_params,
    )


ATOM_METHODS = [
    ("Taylor", "taylor"),
    ("Fitted quadratic", "fitted"),
    ("Split + fitted", "split_fitted"),
    ("Adaptive fitted", "adaptive"),
]

## High-Count Plane-Wave Atom-Slice Benchmark

The large plane-wave atom-slice benchmark has moved to `examples/atoms/plane_wave_atom_slice_gaussian_benchmark.ipynb`. Keep this notebook focused on method comparison; use the standalone benchmark notebook for the 50k-Gaussian timing and longer-propagation diffraction views.

## Single Centered Atom Stress Test

This is the worst thin-slice case for Taylor expansion because the beam is centered on a sharp atom core. A useful unified method should make this case much better without manually special-casing atoms.

In [ ]:
atom_voltage = 200e3
single_extent = 3.0
single_pixel = 2 * single_extent / GRID_SHAPE
single_grid = Detector(
    z=0.0,
    pixel_size=(single_pixel, single_pixel),
    shape=(GRID_SHAPE, GRID_SHAPE),
)
single_atom_xyz = jnp.array([[0.0, 0.0, 0.0]], dtype=jnp.float64)
single_cutoff_radius = single_pixel / 3.0
single_propagation = 2.0
ATOM_DZ_SWEEP = [0.005, 0.02, 0.05]
ATOM_WAIST_SWEEP = [0.45, 0.7, 1.0]

single_atom_records = []
single_atom_fields = {}
for waist in ATOM_WAIST_SWEEP:
    for dz in ATOM_DZ_SWEEP:
        beam = make_gaussian(
            x=0.0,
            y=0.0,
            z=0.0,
            voltage=atom_voltage,
            waist_x=waist,
            waist_y=waist,
            wavelength_unit="angstrom",
        )
        action_fn = make_atom_action_for_beam(
            beam,
            single_atom_xyz,
            dz,
            single_cutoff_radius,
        )
        case_name = f"single Au waist={waist:g} dz={dz:g}"
        records, fields = run_action_case(
            case_name,
            beam,
            single_grid,
            action_fn,
            single_propagation,
            ATOM_METHODS,
        )
        for rec in records:
            rec["domain"] = "single_atom"
            rec["sweep_value"] = dz
            rec["waist"] = waist
        single_atom_records.extend(records)
        single_atom_fields[case_name] = fields

print_records(single_atom_records)

In [ ]:
def plot_atom_heat(records, method_name, title):
    rows = [r for r in records if r["method"] == method_name]
    waists = sorted({r["waist"] for r in rows})
    dzs = sorted({r["sweep_value"] for r in rows})
    image = np.full((len(waists), len(dzs)), np.nan)
    for r in rows:
        image[waists.index(r["waist"]), dzs.index(r["sweep_value"])] = r["field_error"]

    fig, ax = plt.subplots(figsize=(5.2, 3.8))
    log_image = np.log10(np.maximum(image, 1e-16))
    im = ax.imshow(log_image, origin="lower", aspect="auto")
    ax.set_xticks(range(len(dzs)), [f"{v:g}" for v in dzs])
    ax.set_yticks(range(len(waists)), [f"{v:g}" for v in waists])
    ax.set_xlabel("slice thickness dz (A)")
    ax.set_ylabel("Gaussian waist (A)")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label="log10(aligned field error)")
    fig.tight_layout()
    return fig


plot_atom_heat(single_atom_records, "Taylor", "Single atom Taylor error")
plot_atom_heat(single_atom_records, "Fitted quadratic", "Single atom fitted-quadratic error")
plot_atom_heat(single_atom_records, "Adaptive fitted", "Single atom adaptive fitted error")
plt.show()

## Small Atom Plane Thin-Slice Test

This test keeps the atom scope to thin slices but moves beyond a single isolated core. The grid reference applies the whole projected atom-plane transmission before Fresnel propagation.

In [ ]:
au_lattice_constant = 4.078
au_column_spacing = au_lattice_constant / np.sqrt(2.0)
plane_n_side = 5
plane_atom_xyz = atom_plane(plane_n_side, au_column_spacing)
plane_extent = 0.5 * (plane_n_side - 1) * au_column_spacing + 4.0
plane_pixel = 2 * plane_extent / GRID_SHAPE
plane_grid = Detector(
    z=0.0,
    pixel_size=(plane_pixel, plane_pixel),
    shape=(GRID_SHAPE, GRID_SHAPE),
)
plane_cutoff_radius = plane_pixel / 3.0
plane_propagation = 20.0
PLANE_DZ_SWEEP = [0.005, 0.02, 0.05]
PLANE_WAIST_SWEEP = [0.8, 1.2]

plane_records = []
plane_fields = {}
for waist in PLANE_WAIST_SWEEP:
    for dz in PLANE_DZ_SWEEP:
        beam = make_gaussian(
            x=0.0,
            y=0.0,
            z=0.0,
            voltage=atom_voltage,
            waist_x=waist,
            waist_y=waist,
            wavelength_unit="angstrom",
        )
        action_fn = make_atom_action_for_beam(
            beam,
            plane_atom_xyz,
            dz,
            plane_cutoff_radius,
        )
        case_name = f"5x5 Au plane waist={waist:g} dz={dz:g}"
        records, fields = run_action_case(
            case_name,
            beam,
            plane_grid,
            action_fn,
            plane_propagation,
            ATOM_METHODS,
        )
        for rec in records:
            rec["domain"] = "atom_plane"
            rec["sweep_value"] = dz
            rec["waist"] = waist
        plane_records.extend(records)
        plane_fields[case_name] = fields

print_records(plane_records)

In [ ]:
plot_atom_heat(plane_records, "Taylor", "5x5 atom-plane Taylor error")
plot_atom_heat(plane_records, "Fitted quadratic", "5x5 atom-plane fitted-quadratic error")
plot_atom_heat(plane_records, "Adaptive fitted", "5x5 atom-plane adaptive fitted error")
plt.show()

## Decision Plots

These plots compare all lens and atom cases in the same coordinates. A useful unified method should move down in error at acceptable beam count and runtime. The vertical residual lines mark the proposed residual band for split/refit decisions.

In [ ]:
all_records = lens_records + single_atom_records + plane_records

method_order = [
    "Taylor",
    "Fitted quadratic",
    "Split + fitted",
    "Adaptive fitted",
    "Hermite residual",
]
colors = dict(zip(method_order, plt.rcParams["axes.prop_cycle"].by_key()["color"]))

fig, axes = plt.subplots(1, 3, figsize=(14, 4.0))
for method in method_order:
    rows = [r for r in all_records if r["method"] == method]
    if not rows:
        continue
    color = colors[method]
    axes[0].scatter(
        [r["residual_max_rad"] for r in rows],
        [r["field_error"] for r in rows],
        label=method,
        alpha=0.75,
        color=color,
    )
    axes[1].scatter(
        [r["beams"] for r in rows],
        [r["field_error"] for r in rows],
        alpha=0.75,
        color=color,
    )
    axes[2].scatter(
        [r["time_s"] for r in rows],
        [r["field_error"] for r in rows],
        alpha=0.75,
        color=color,
    )

for x in RESIDUAL_BAND_RAD:
    axes[0].axvline(x, color="k", linestyle="--", linewidth=1)
axes[0].axvline(RESIDUAL_TARGET_RAD, color="k", linestyle="-", linewidth=1.2)

axes[0].set_xlabel("parent fitted residual max (rad)")
axes[1].set_xlabel("output beam count")
axes[2].set_xlabel("method runtime (s)")
for ax in axes:
    ax.set_yscale("log")
    ax.set_ylabel("aligned field error")
    ax.grid(True, alpha=0.3)
axes[0].set_xscale("symlog", linthresh=1e-4)
axes[1].set_xscale("log")
axes[2].set_xscale("log")
axes[0].legend(fontsize=8)
fig.suptitle("Unified correction decision space")
fig.tight_layout()
plt.show()

In [ ]:
def best_by_case(records):
    grouped = defaultdict(list)
    for r in records:
        grouped[r["case"]].append(r)
    winners = []
    for case, rows in grouped.items():
        best = min(rows, key=lambda r: r["field_error"])
        winners.append(best)
    return sorted(winners, key=lambda r: (r["domain"], r["case"]))


print("Best method per case by aligned field error")
print_records(best_by_case(all_records), sort_keys=("domain", "case"))

## How To Read The Result

The notebook is designed to answer two questions.

First, if sampled fitted quadratics consistently beat Taylor for centered atoms and remain competitive for smooth lenses, then they are the right shared base operator.

Second, if split/refit only helps above the residual band near `0.1-0.3 rad`, then residual-driven splitting is the right adaptive policy. If split/refit helps below that band, lower the threshold. If it does not help above that band, the child fit geometry or support radius needs work before claiming a robust atom method.

The paper-level claim should be made only from cases where the fitted/adaptive method improves atom thin-slice errors and the error decreases as residual decreases.